# Tech Question AI Assistant

A tool that takes a technical question, and responds with an explanation.

In [1]:
# imports

import os
import requests
from dotenv import load_dotenv
from bs4 import BeautifulSoup
from IPython.display import Markdown, display, update_display
from openai import OpenAI

In [2]:
# constants

MODEL_GPT = 'gpt-4o-mini'
MODEL_LLAMA = 'llama3.2'

OLLAMA_API = "http://localhost:11434/v1"
HEADERS = {"Content-Type": "application/json"}

In [3]:
# set up environment

load_dotenv(override=True)
api_key = os.getenv('OPENAI_API_KEY')

if api_key and api_key.startswith('sk-proj-') and len(api_key)>10:
    print("API key looks good so far")
else:
    print("There might be a problem with your API key? Please visit the troubleshooting notebook!")
    

API key looks good so far


In [4]:
# here is the system prompt and payloads;

system_prompt = """
You are an expert on LLMs and writing python code. You are able to answer complex questions with
detailed answers and explain what every line of code does. You can refactor the code when asked.
"""

In [5]:
# Function to get answer, with streaming

def llm_copilot(question, model):
    if 'llama' in model.lower():
        openai = OpenAI(base_url=OLLAMA_API, api_key='ollama')
    else:
        openai = OpenAI()
        
    stream = openai.chat.completions.create(
        model=model,
        messages=[
            {"role": "system", "content": system_prompt},
            {"role": "user", "content": question}
          ],
        stream=True
    )
    response = ""
    display_handle = display(Markdown(""), display_id=True)
    for chunk in stream:
        response += chunk.choices[0].delta.content or ''
        response = response.replace("```","").replace("markdown", "")
        update_display(Markdown(response), display_id=display_handle.display_id)

In [6]:
# Ask question
question = """
Create python code to read an array of data in a json file using Dataset.from_dict

json file example format:
[{
  "_id": "1888de0d-6b11-4fd2-a9e2-46c83535bc9c",
  "content": {
    "Title": "Hex Gen - Gygax Challenge",
    "Subtitle": "I’m planning on starting an Old School Essentials game and needed to create a hex map for it. While looking up techniques to do this I came across this video series by Games from the Front that inspired me to use this approach and wanted to share it with my fellow TTRPG enthusiasts: Old School Essentials.",
    "Content": "# Old School Gamer Blog\n\n* * *\n\n## Ramblings from old-school gamers\n\nHome Tools Blog Staff\n\n# Hex Gen - Gygax Challenge\n\n25 Jul 2025 \\- Cpt. Redbeard\n\nI’m planning on starting an Old School Essentials game and needed to create a\nhex map for it. While looking up techniques to do this I came across this\nvideo series by Games from the Front that inspired me to use this approach and\nwanted to share it with my fellow TTRPG enthusiasts: Old School Essentials.\n\nThis approach can be used for any OSR (Old School Renaissance or Old School\nRevival) game like the original AD&D Dungeon Master’s Guide (DMG) 1e, Dungeon\nCrawl Classics (DCC), Shadowdark, Deathbringer, Grave, Iron Halberd, Knave,\nBasic Fantasy, and many others based on the original DnD rules.\n\nThe main inspiration for this “world building” process comes from The Gygax 75",
    "language": "No language found."
  },
  "platform": "jstoops.github.io",
  "author_id": "bdca1327-a403-4486-bff0-fb9d7df40467",
  "author_full_name": "John Stoops",
  "link": "https://jstoops.github.io/2025/07/25/osr-hex-gen-gygax-challenge.html"
},
{
  "_id": "720fd719-3796-49cf-a8b9-020b21b5f76f",
  "content": {
    "Title": "Hugging Face Launch Open Source Programmable Robot!",
    "Content": "Games like Skyrim and 7 Days to Die changed how we gamers and the gaming industry approached gaming by opening them up to the community for\\xa0modding.This provided endless opportunities for us to customize the games we play with our friends to our unique gaming group and now we can do the same with\\xa0robots!Reachy Mini Robot from Hugging\\xa0FaceReachy MiniFor under 300 bucks you can to buy a desktop robot from Hugging Face that sits next to your keyboard plugged into your computer.Out-of-the-box this 11 inch tall robot wiggles, twitches antenna at you, tracks your face, and even dances. It supports vision, text and speech that you can train your AI model to interpret, make decisions based on what it sees and hears, hold conversations, and interact with its surroundings.The best thing about this robot is that it is completely open source."
  },
  "platform": "medium",
  "author_id": "bdca1327-a403-4486-bff0-fb9d7df40467",
  "author_full_name": "John Stoops",
  "link": "https://medium.com/@john.stoops/hugging-face-launch-open-source-programmable-robot-faad48c97d5f?source=rss-ac55c8d08c9b------2"
}]
"""

print(llm_copilot(question, MODEL_GPT))

To read an array of data from a JSON file and convert it into a format suitable for using with `Dataset.from_dict`, you can follow these steps:

1. **Load the JSON data**: Use the built-in `json` module to read the data from the file.
2. **Transform the data**: Prepare the data for `Dataset.from_dict` by flattening it if necessary.
3. **Create the dataset**: Use `Dataset.from_dict` to create a dataset from the flattened dictionary.

Here’s a sample Python code implementation that demonstrates these steps:

python
import json
from datasets import Dataset

def load_json_dataset(file_path):
    """
    Load a JSON array from a file and convert it to a Dataset.
    
    Args:
        file_path (str): The path to the JSON file.

    Returns:
        Dataset: A Hugging Face Dataset object created from the JSON data.
    """
    # Step 1: Load the JSON data from the specified file.
    with open(file_path, 'r') as file:
        data = json.load(file)

    # Step 2: Prepare the data for Dataset.from_dict.
    # Create lists to hold the values of interest.
    ids = []
    titles = []
    subtitles = []
    contents = []
    platforms = []
    author_ids = []
    author_names = []
    links = []

    # Iterate through the loaded JSON data.
    for item in data:
        ids.append(item['_id'])
        titles.append(item['content']['Title'])
        subtitles.append(item['content'].get('Subtitle', ''))  # Use .get to avoid KeyError
        contents.append(item['content']['Content'])
        platforms.append(item['platform'])
        author_ids.append(item['author_id'])
        author_names.append(item['author_full_name'])
        links.append(item['link'])
    
    # Step 3: Create a dictionary from the collected lists.
    dataset_dict = {
        'id': ids,
        'title': titles,
        'subtitle': subtitles,
        'content': contents,
        'platform': platforms,
        'author_id': author_ids,
        'author_full_name': author_names,
        'link': links,
    }

    # Create and return a Dataset from the created dictionary.
    return Dataset.from_dict(dataset_dict)

# Example usage:
# Assuming you have your JSON data saved in 'data.json'
dataset = load_json_dataset('data.json')
print(dataset)


### Explanation of the Code:
- **Imports**: We import the `json` module to read the JSON file and `Dataset` from the `datasets` library to create a dataset.
- **Function Definition**: We define a function `load_json_dataset` that takes the `file_path` as an argument.
- **Load JSON**: We open the specified JSON file in read mode and load its contents using `json.load()`.
- **Data Preparation**: We initialize lists to capture data for each significant field in our items:
  - For each item in the JSON array, we extract the desired fields and append them to the corresponding lists. 
  - We use `item['content'].get('Subtitle', '')` to safely access the subtitle, ensuring that if it's missing, we don't raise an error.
- **Create a Dictionary**: After processing all items, we create a dictionary where each key corresponds to a field, and the value is a list of all entries for that field.
- **Create Dataset**: Finally, we use `Dataset.from_dict` to convert the dictionary into a dataset and return it.

### Usage:
To use the function, simply call it with the path to your JSON file. This will read the data and create a Hugging Face dataset from it, which can then be used for further processing, training, or analysis.

None
